# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [ ]:
# Instalación de dependencias
# Se instalan las dependencias dentro de un entorno controlonado (.venv) con
# pip install -q --upgrade pypdf gradio openai python-dotenv 

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [3]:
from pypdf import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# Probemos la función con un archivo PDF específico, imprimamos los primeros 500 caracteres
text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(text[:500])


Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [ ]:
import os, openai
from dotenv import load_dotenv
from openai import OpenAI

# Es el modelo que estamos utilizando
MODELO = "nvidia/nemotron-3-super-120b-a12b"

load_dotenv() # Cargamos las variables de entorno desde el archivo .env

openai.api_key = os.getenv("NVIDIA_API_KEY")
print("API key loaded:", openai.api_key is not None)

openai_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)


API key loaded: True


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [ ]:
#Libreria para la interfaz de usuario
import gradio as gr

# Construcción del system prompt basado en el texto del documento
def build_system_prompt(document_text):
    return (
        "Eres un experto en el paper 'Attention Is All You Need'. "
        "Responde ÚNICAMENTE con base en el documento delimitado abajo; no uses conocimiento externo. "
        "Si la respuesta no está en el documento, responde exactamente: "
        "'No encuentro esa información en el documento'.\n\n"
        "<documento>\n"
        f"{document_text}\n"
        "</documento>"
    )

# Conversión del historial de gradio (tiene formato propio) a mensajes para el modelo con roles user y assistant
def history_to_messages(history):
    # Gradio reciente entrega dicts {"role", "content"}; se soporta también el formato antiguo de tuplas
    messages = []
    for item in history:
        # Si esta en un formato de diccionario {"role", "content"}, usamos de esta manera
        # user -> hablo el "humano"
        # assistant -> hablo la IA
        if isinstance(item, dict):
            messages.append({"role": item["role"], "content": item["content"]})
        # Si no esta en un formato de diccionario asumimos que es una tupla (user_message, assistant_message) y resolvemos
        else:
            user_message, assistant_message = item
            messages.append({"role": "user", "content": user_message})
            if assistant_message:
                messages.append({"role": "assistant", "content": assistant_message})
    return messages

# Función principal de chat que el mensaje del usuario , el historial de conversación y el documento

def chat_con_documento(message, history, document_text):
    # Se le dan las reglas de juego al modelo incialmente
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    # Se agrega el historial de conversación previo al mensaje actual
    messages += history_to_messages(history)
    # Se le pasa el mensaje como tal del usuario
    messages.append({"role": "user", "content": message})

    # LLamada al API de OpenAI para generar la respuesta del modelo
    stream = openai_client.chat.completions.create(
        model=MODELO, 
        messages=messages, #historial de conversación y mensaje actual del usuario
        temperature=1.0, #el modelo es más creativo
        top_p=0.95, #el modelo considera solo el top_p de las opciones más probables de palabras (evita respuestas raras)
        stream=True, #Se va mandando la respuesta poco a poco
    )

    # Se inicializa la variable que contendrá la respuesta completa del modelo
    # Y se va armando en pedacitos
    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta # Se va entregando la respuesta poco a poco al usuario


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [20]:
# Extraemos el texto del PDF con la función del Paso 1
document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
num_paginas = len(PdfReader("attention_is_all_you_need.pdf").pages)
print(f"Páginas: {num_paginas} | Caracteres: {len(document_text)}")

demo = gr.ChatInterface(
    fn=chat_con_documento, # Función que maneja la interacción con el documento
    title="Asistente de 'Attention Is All You Need'",
    description="Hazle preguntas al paper. Responde solo con base en el documento.",
    # Se pasa el texto del documento como un input adicional oculto
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?", document_text],
        ["¿Qué es el mecanismo de atención?", document_text],
        ["¿Cuántas capas tiene el encoder del modelo base?", document_text],
        ["¿Quiénes son los autores del paper?", document_text],
        ["¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?", document_text],
        ["¿Qué es GPT-4?", document_text],
    ],
)

demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

Páginas: 15 | Caracteres: 39585
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*

**El modelo respondió:** Transformer

2. *"¿Qué es el mecanismo de atención?"*

**El modelo respondió:** Una función de atención puede describirse como un mapeo de una consulta y un conjunto de pares clave-valor a una salida, donde la consulta, las claves, los valores y la salida son todos vectores. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor se calcula mediante una función de compatibilidad de la consulta con la correspondiente clave.


3. *"¿Cuántas capas tiene el encoder del modelo base?"*

**El modelo respondió:** 6 Layers. (The encoder is composed of a stack of N = 6 identical layers.) -> Se le descuadró el idioma

4. *"¿Quiénes son los autores del paper?"*

**El modelo respondió:** Los autores del paper son:
• Ashish Vaswani
• Noam Shazeer
• Niki Parmar
• Jakob Uszkoreit
• Llion Jones
• Aidan N. Gomez
• tukasz Kaiser
• Illia Polosukhin
(Los asteriscos indican contribución igual; los símbolos † y f indican trabajo realizado mientras estaban en Google Brain y Google Research, respectivamente)

5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

**El modelo respondió:** 28.4 BLEU
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

**El modelo respondió:** No encuentro esa información en el documento 

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

El modelo sí respeta la instrucción de ceñirse al doc.
 

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?

   **R:** Está limitado por la ventana de contexto: documentos largos no caben o se truncan. Además, cada pregunta reenvía todo el texto, lo que aumenta costo, latencia y consumo de tokens, y mucho texto irrelevante puede distraer al modelo.

2. ¿Por qué este ejercicio todavía no implementa RAG?

   **R:** Porque no hay recuperación: no se divide el documento en fragmentos ni se buscan los más relevantes para cada pregunta. Siempre se inyecta el documento completo, mientras que RAG envía solo los fragmentos pertinentes.

3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?

   **R:** Haciendo preguntas cuya respuesta no está en el paper (como "¿Qué es GPT-4?") y verificando que responda "No encuentro esa información en el documento". También exigiendo citas textuales o de página y contrastándolas con el PDF, o usando un documento modificado con datos inventados y viendo si el modelo los repite.

4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

   **R:** Prompt injection (instrucciones maliciosas ocultas en el documento que alteran el comportamiento del modelo), fuga de datos sensibles o privados hacia el modelo/proveedor, y desbordar el contexto con archivos enormes o manipulados.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)